Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np

Step 2 — Load Feature-Engineered Dataset

In [2]:
df = pd.read_csv(
    "../data/processed/upi_transactions_feature_engineered.csv"
)

In [3]:
print("Shape:", df.shape)

Shape: (10000, 29)


Step 3 — Check the Dataset

In [4]:
df.head()

,Transaction_ID,Timestamp,Sender_ID,Receiver_ID,Amount,Merchant_Category,Transaction_Type,Location,Device_Type,UPI_Channel,...,Day_of_Week.1,Is_Night,User_Avg_Amount,Amount_Deviation,Amount_Difference,Transactions_Per_Hour,Time_Since_Last_Transaction,Recipient_Frequency,Device_Change,Location_Change
0,TXN102315,2026-02-12 06:05:18,U1000,R1034,805.48,Grocery,P2M,Noida,iOS,UPI Lite,...,3,0,3120.61,0.258116,-2315.13,1,NaN,1,0,0
1,TXN103673,2026-03-09 03:16:53,U1000,R1044,325.77,Fuel,Online Purchase,Surat,Android,UPI App,...,0,1,3120.61,0.104393,-2794.84,1,35831.583333,1,1,1
2,TXN105082,2026-04-03 18:16:39,U1000,R1520,1541.69,E-commerce,P2M,Surat,Android,UPI App,...,4,0,3120.61,0.494035,-1578.92,1,36899.766667,1,0,0
3,TXN106301,2026-04-26 01:34:52,U1000,R1410,948.36,Fuel,P2P,Noida,iOS,UPI App,...,6,1,3120.61,0.303902,-2172.25,1,32118.216667,1,1,1
4,TXN107005,2026-05-08 14:35:53,U1000,R1129,1492.54,Shopping,Online Purchase,Bengaluru,Android,USSD,...,4,0,3120.61,0.478285,-1628.07,1,18061.016667,1,1,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Transaction_ID               10000 non-null  object 
 1   Timestamp                    10000 non-null  object 
 2   Sender_ID                    10000 non-null  object 
 3   Receiver_ID                  10000 non-null  object 
 4   Amount                       10000 non-null  float64
 5   Merchant_Category            10000 non-null  object 
 6   Transaction_Type             10000 non-null  object 
 7   Location                     10000 non-null  object 
 8   Device_Type                  10000 non-null  object 
 9   UPI_Channel                  10000 non-null  object 
 10  Hour_of_Day                  10000 non-null  int64  
 11  Day_of_Week                  10000 non-null  object 
 12  Historical_Spend_Pattern     10000 non-null  float64
 13  Spend_Deviation_R

Step 4 — Select Features for IQR

In [6]:
iqr_features = [
    "Amount",
    "Amount_Deviation",
    "Transactions_Per_Hour"
]

Step 5 — Create IQR Function

In [7]:
def detect_iqr_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    )

    return Q1, Q3, IQR, lower_bound, upper_bound, outlier

Step 6 — Apply IQR to Amount

In [8]:
Q1, Q3, IQR, lower, upper, outlier = detect_iqr_outliers(
    df,
    "Amount"
)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

Q1: 497.595
Q3: 1690.8025
IQR: 1193.2075
Lower Bound: -1292.21625
Upper Bound: 3480.61375


Step 7 — Create Amount IQR Alert

In [9]:
df["IQR_Amount_Alert"] = outlier.astype(int)

In [10]:
df["IQR_Amount_Alert"].value_counts()

IQR_Amount_Alert
0    9204
1     796
Name: count, dtype: int64

In [11]:
Q1, Q3, IQR , Lower, upper, outlier = detect_iqr_outliers (
    df,"Amount_Deviation"
)

df["IQR_Deviation_Alert"] = outlier.astype(int)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

Q1: 0.5027881823930008
Q3: 1.2520778218035042
IQR: 0.7492896394105034
Lower Bound: -1292.21625
Upper Bound: 2.3760122809192596


In [12]:
df["IQR_Deviation_Alert"].value_counts()

IQR_Deviation_Alert
0    9530
1     470
Name: count, dtype: int64

Step 9 — Apply IQR to Transaction Frequency

In [13]:
Q1, Q3, IQR, Lower, upper, outlier = detect_iqr_outliers (
    df,"Transactions_Per_Hour"
)
df["IQR_Frequency_Alert"] = outlier.astype(int)

print("Q1:", Q1)
print("Q2:", Q3)
print("IQR:", IQR)
print("Lower Bound", lower)
print("Upper Bound", upper)

Q1: 1.0
Q2: 1.0
IQR: 0.0
Lower Bound -1292.21625
Upper Bound 1.0


In [14]:
df["IQR_Frequency_Alert"].value_counts()

IQR_Frequency_Alert
0    9986
1      14
Name: count, dtype: int64

Step 10 — Combine IQR Alerts

In [15]:
df["IQR_Alert_Count"] = (
    df["IQR_Amount_Alert"] + df["IQR_Deviation_Alert"] + df["IQR_Frequency_Alert"]
)

Step 11 — Create Overall IQR Alert

In [16]:
df["IQR_Alert"] = (df["IQR_Alert_Count"] > 0
                  ).astype(int)

Step 12 — Analyze Results

In [17]:
print(df["IQR_Alert"].value_counts())

IQR_Alert
0    9011
1     989
Name: count, dtype: int64


Percentage

In [18]:
print(
    df["IQR_Alert"]
    .value_counts(normalize =True)
    .mul(100)
    .round(2)
)

IQR_Alert
0    90.11
1     9.89
Name: proportion, dtype: float64


Step 13 — View Flagged Transactions

In [19]:
flagged_transactions = df[
    df["IQR_Alert"] == 1
]

flagged_transactions[
    [
        "Transaction_ID",
        "Amount",
        "Amount_Deviation",
        "Transactions_Per_Hour",
        "IQR_Alert_Count"
    ]
].head(20)


,Transaction_ID,Amount,Amount_Deviation,Transactions_Per_Hour,IQR_Alert_Count
5,TXN108282,13609.82,4.361269,1,2
24,TXN101912,2128.81,2.429213,1,1
35,TXN100772,1673.97,2.543532,1,1
53,TXN100844,550.70,0.753242,2,1
54,TXN100845,854.64,1.168968,2,1
67,TXN109549,4887.30,1.969737,1,1
76,TXN107988,3039.17,2.632467,1,1
87,TXN101174,1477.82,2.642834,1,1
95,TXN102442,11454.70,4.124121,1,2
113,TXN103903,1032.83,2.589720,1,1


Step 14 - Compare With Synthetic Ground Truth

In [20]:
comparision = pd.crosstab(
    df["Anomaly_Flag"],
    df["IQR_Alert"],
    rownames =["Actual Anomaly"],colnames=["IQR Alerts"])
print(comparision)

IQR Alerts         0    1
Actual Anomaly           
0               8865  785
1                146  204


Step 15 Save IQR Results

In [22]:
df.to_csv("../data/processed/""upi_transactions_iqr_results.csv",index=False)